# Choi et al. (2025): the earlier attempt's findings, checked on DANDI:002014

Attempt `choi-2025-02`, made blind, is compared with `choi-2025-01` in the reconciliation block of
`attempts/choi-2025-02.yaml` (docs/protocol.md, "Repeat attempts"). `choi-2025-01` was made on version
0.261012.0921. Findings 6, 11 and 12 of `choi-2025-01` are reported again by `choi-2025-02` (its findings 3,
9 and 1) and are not checked again here, except for the part of finding 6 that `choi-2025-02` does not state.
This notebook checks the others directly on
[DANDI:002014](https://dandiarchive.org/dandiset/002014/0.261120.0528), version 0.261120.0528, streaming the
one NWB file with remfile. It uses the entry's helper module `choi2025.py` to open the file and to read the
events, trials, units and LFP.

In [ ]:
import numpy as np
import pandas as pd
from scipy import signal

import choi2025 as c

pd.set_option("display.width", 200)
print("dandiset", c.DANDISET_ID, "version", c.DANDISET_VERSION)
assets = c.list_assets()
f = c.open_nwb(assets[0])
events = c.load_events(f)
trials = c.trial_table(f, events)
es = f["acquisition/electrical_series"]
t_raw0 = float(es["starting_time"][()])
fs_raw = float(es["starting_time"].attrs["rate"])
conv = float(es["data"].attrs["conversion"])
lfp_t, lfp_x = c.load_lfp(f)

## Data check

In [ ]:
print("assets:", [(a["path"], a["size"]) for a in assets])
print("session start:", c.text(f["session_start_time"][()]))
el = f["general/extracellular_ephys/electrodes"]
print("electrodes:", [(c.text(n), c.text(l)) for n, l in zip(el["channel_name"][()], el["location"][()])])
print("raw ephys:", es["data"].shape, "electrodes", es["electrodes"][()].tolist(), "starting_time", t_raw0, "rate", fs_raw)
print("trials:", len(trials), "| successful large", int(trials.large_success.sum()), "small", int(trials.small_success.sum()))
print("LFP:", len(lfp_x), "samples,", round(lfp_t[0], 2), "to", round(lfp_t[-1], 2), "s")

## Earlier finding 1: clock of the raw ephys and the spikes

> The raw ephys and the spike times are not on the clock of the trials, events, LFP and FSCV table. The raw ephys is stored with a starting time of 0 s and a rate, and the spike times count from its first sample. Locating the raw c7b minus c5c signal in the stored LFP puts raw time 0 at 11812.11 s on the trial clock (correlation 0.98 to 1.00 at eight points, drift about 4 ms per hour). Without this correction spikes cannot be aligned to the task. The dandiset description says all data are expressed relative to the start of the Neuralynx recording, which the data contradict.

The raw c7b minus c5c, reduced to 1 kHz, is compared with the stored LFP in 1 to 8 Hz around the first and
the last row of the trials table, over lags of 2 s either way.

In [ ]:
sos = signal.butter(4, (1, 8), btype="bandpass", fs=1000, output="sos")
rows = []
for tr in [0, len(trials) - 1]:
    m = trials.start[tr]
    L = c.lfp_segment(lfp_t, lfp_x, m, 10, 10)
    ia, ib = int((m - 12 - t_raw0) * fs_raw), int((m + 12 - t_raw0) * fs_raw)
    raw = es["data"][ia:ib, :].astype(float) * conv
    bd = signal.decimate(signal.decimate(raw[:, 0] - raw[:, 1], 8), 4)
    td = t_raw0 + ia / fs_raw + np.arange(len(bd)) * 32 / fs_raw
    R = np.interp(np.arange(m - 12, m + 12, 0.001), td, bd)
    Lf, Rf = signal.sosfiltfilt(sos, L), signal.sosfiltfilt(sos, R)
    lags = np.arange(-2000, 2001)
    cc = np.array([np.corrcoef(Lf, Rf[2000 + k:2000 + k + len(Lf)])[0, 1] for k in lags])
    k = int(np.argmax(cc))
    rows.append(dict(row=tr, time_s=round(m, 1), best_lag_ms=int(lags[k]), r_best=round(cc[k], 3), r_lag0=round(cc[2000], 3)))
print(pd.DataFrame(rows).to_string(index=False))
print("spike times, thresholded unit 1 and Plexon unit:",
      [(k, round(float(v.min()), 2), round(float(v.max()), 2)) for k, v in
       [("1", c.load_units(f, "processing/ecephys/thresholded_units")["1"]), ("ch75#1", c.load_units(f, "units")["ch75#1"])]])
print("session description:", c.text(f["session_description"][()]))

The raw ephys now starts at 4612.10 s on the session clock, and it lines up with the stored LFP at a lag of
1 ms at both ends of the session. The spike times fall within the raw recording. The problem is gone, as
`choi-2025-02` also found (its finding 2).

## Earlier finding 2: absolute times

> The absolute times disagree. On the trial clock the raw samples run from 12:06:44 to 17:12:20, while the Neuralynx header gives the recording as opened at 08:49:52 (the session start) and closed at 15:12:24, two hours before the raw samples end. The header span (22952 s) is also longer than the raw samples (18336 s).

In [ ]:
start = pd.Timestamp(c.text(f["session_start_time"][()]))
t_raw1 = t_raw0 + es["data"].shape[0] / fs_raw
print(f"raw samples: {t_raw0:.2f} to {t_raw1:.2f} s, {t_raw1 - t_raw0:.0f} s, i.e. "
      f"{(start + pd.Timedelta(seconds=t_raw0)).time()} to {(start + pd.Timedelta(seconds=t_raw1)).time()}")
print("header opened:", [c.text(x) for x in el["recording_opened"][()]])
print("header closed:", [c.text(x) for x in el["recording_closed"][()]])
closed = pd.Timestamp("2024-09-27 15:12:24", tz=start.tz)
print(f"header span {(closed - start).total_seconds():.0f} s; raw end before header close by "
      f"{(closed - start).total_seconds() - t_raw1:.1f} s")

The raw samples now end at 15:12:20, 4 s before the header's close, and they start 4612 s after the
recording was opened, which accounts for the header span being longer than the samples. The absolute times
agree. The problem is gone.

## Earlier finding 3: spike times against the raw trace

> The stored spike times of the thresholded units are about 7.1 ms after the spike troughs in raw c5d (in one 300 s segment of 1633 spikes, per-spike trough lags of -7.4 to -6.9 ms from the 10th to the 90th percentile, 98.8% within 0.5 ms of -7.1 ms).

The raw c5d trace, band-passed 300 to 6000 Hz, is averaged at the stored times of thresholded unit 1 in 60 s
at the start, middle and end of the recording, and the trough of each spike is located within 10 ms either way.

In [ ]:
s_thr = c.load_units(f, "processing/ecephys/thresholded_units")["1"]
sos_hp = signal.butter(4, [300, 6000], btype="bandpass", fs=fs_raw, output="sos")
rows = []
for q in [0.05, 0.5, 0.95]:
    t_a = s_thr[int(q * len(s_thr))]
    ia = int((t_a - t_raw0) * fs_raw); ib = ia + int(60 * fs_raw)
    hp = signal.sosfiltfilt(sos_hp, es["data"][ia:ib, 2].astype(float) * conv)
    sp = s_thr[(s_thr > t_raw0 + (ia + 400) / fs_raw) & (s_thr < t_raw0 + (ib - 400) / fs_raw)]
    idx = np.round((sp - t_raw0) * fs_raw).astype(int) - ia
    seg = np.array([hp[i - 320:i + 321] for i in idx])
    lag_ms = (np.argmin(seg, 1) - 320) / fs_raw * 1000
    avg = seg.mean(0)
    rows.append(dict(from_s=round(t_a, 1), spikes=len(sp), mean_trough_ms=round((np.argmin(avg) - 320) / fs_raw * 1000, 3),
                     lag_p10_ms=round(np.percentile(lag_ms, 10), 3), lag_p90_ms=round(np.percentile(lag_ms, 90), 3),
                     within_0p5ms_of_mean=round(np.mean(np.abs(lag_ms - (np.argmin(avg) - 320) / fs_raw * 1000) < 0.5), 3)))
print(pd.DataFrame(rows).to_string(index=False))
w = c.thresholded_waveforms(f, 1).mean(0)
print(f"stored waveforms: trough at sample {np.argmin(w)} of {len(w)}, i.e. {np.argmin(w) / 32.0:.3f} ms after the first sample")

The raw c5d trough now falls 0.000 to 0.344 ms after the stored spike times on average, at the start, middle
and end of the recording, close to the 0.312 ms at which the stored waveforms have their trough (sample 10),
with 0.997 to 1.000 of the spikes within 0.5 ms of that average. The 7.1 ms lag is gone. The problem is gone.

## Earlier finding 4: the LFP's description and scale

> The series "bipolar_lfp_series" is described as beta-band LFP but is broadband with line noise removed. Its values are already in volts, while it carries the raw int16 conversion factor (3.05e-8), which would scale them about 3e7 times too small.

In [ ]:
g = f["processing/ecephys/FilteredEphys/bipolar_lfp_series"]
print("description:", c.text(g.attrs["description"]))
print("unit:", c.text(g["data"].attrs["unit"]), "| conversion:", g["data"].attrs["conversion"])
print(f"median |value|: {np.nanmedian(np.abs(lfp_x)) * 1e6:.1f} uV after conversion")

The series is no longer called beta-band, its conversion is 1.0, and its values are of the size of an LFP in
volts. The problem is gone. The description now says the preprocessing is not documented, which
`choi-2025-02` reports as its finding 4.

## Earlier finding 5: units of the waveforms and the raw FSCV current

> The thresholded spike waveforms are stated in volts but hold microvolts (mean trough about -56), and the raw FSCV current is stated in amperes but holds nanoamperes (values around 1000).

In [ ]:
wf = f["processing/ecephys/thresholded_units/waveforms"]
print("waveforms unit:", c.text(wf.attrs["unit"]), f"| mean waveform trough {w.min():.3e}")
fr = f["acquisition/fscv_response_series/data"]
x = fr[:214 * 100]
print("FSCV current unit:", c.text(fr.attrs["unit"]), "| conversion", fr.attrs["conversion"],
      f"| stored values {x.min():.1f} to {x.max():.1f}, i.e. {x.min() * fr.attrs['conversion']:.2e} to "
      f"{x.max() * fr.attrs['conversion']:.2e} A")

The waveforms now hold volts (mean trough -5.6e-05 V, -56 uV), and the raw FSCV current carries a conversion
of 1e-9, so the stored values in nanoamperes give amperes. The problem is gone.

## Earlier finding 6: units and quality flag of the trial-aligned FSCV table

> The columns of the trial-aligned FSCV table (da, ph, m, iox) state no units, and the `good` column is true for every trial.

The missing units are reported again by `choi-2025-02` (its finding 3). The `good` column is checked here.

In [ ]:
good = f["processing/fscv/trial_aligned_fscv_table/good"][()]
print("good:", int(good.sum()), "of", len(good), "true |", c.text(f["processing/fscv/trial_aligned_fscv_table/good"].attrs["description"]))
for k in ["da", "ph", "m", "iox"]:
    print(k, "|", c.text(f[f"processing/fscv/trial_aligned_fscv_table/{k}"].attrs["description"]))

The four columns still state no units, and `good` is still true for all 1411 trials, with a description
saying only that it comes from the source data.

## Earlier finding 7: subject and session descriptions

> The subject description ("Two adult female rhesus monkeys were used for in vivo validation of the s-µIP device"), the session and experiment descriptions and the keyword "micro-invasive neural implant" describe the lab's micro-invasive probe study, not this paper's chamber.

In [ ]:
print("subject:", c.text(f["general/subject/description"][()]))
print("session:", c.text(f["session_description"][()]))
print("experiment:", c.text(f["general/experiment_description"][()]))
print("keywords:", [c.text(k) for k in f["general/keywords"][()]])

The subject, session and experiment descriptions and the keywords now describe this paper's chamber and
session. The problem is gone.

## Earlier finding 8: repeated and unsorted events

> The event table stores each event once per overlapping 60 s trial window, about five copies of each (7221 stored trial-start events for 1412 distinct ones), and its event times are not sorted. The Plexon units table's spike times are not sorted either.

In [ ]:
ev = f["processing/events/events_table"]
times, idx = ev["event_times"][()], ev["event_times_index"][()]
unsorted, repeated, a = 0, 0, 0
for b in idx:
    t = times[a:b]; unsorted += int(np.sum(np.diff(t) < 0)); repeated += len(t) - len(np.unique(t)); a = b
print("stored events:", len(times), "| trial-start events (code 128):", len(events["intended trial start"]),
      "| repeated times within a label:", repeated, "| decreasing steps within a label:", unsorted)
plx = c.load_units(f, "units")
print("Plexon units:", {k: (len(v), int(np.sum(np.diff(v) < 0))) for k, v in plx.items()}, "(spikes, decreasing steps)")

Each event is now stored once and in time order (1412 trial-start events), and the Plexon spike times are
sorted. The problem is gone.

## Earlier finding 9: trial tags and trial types

> The trial tags are all "initial cue display" and carry no trial type. Reward size and outcome can be rebuilt from the events, and they then match the paper's Zenodo trial lists except trial 1411, which the events mark as a successful large reward trial and the list leaves out.

`choi-2025-02` reports the match with the Zenodo lists and trial 1411 (its finding 6). The tags are checked here.

In [ ]:
tags = pd.Series([c.text(x) for x in f["intervals/trials/tags"][()]])
print(tags.value_counts().to_string())
print("trials table columns:", c.text(f["intervals/trials"].attrs["colnames"]))

The tags are still "initial cue display" for all 1411 rows, and the trials table has no column for the
trial type or outcome; they are rebuilt from the events, as before.

## Earlier finding 10: target position and reward size

> The paper says the reward size depends on whether the target is at the top or bottom of the screen. The events code left and right conditions, and both reward sizes occur on both sides, so the target position that sets the reward is not in the dandiset.

Each trial is crossed with the events in its span: the left and right conditions and the reward
conditions (codes 41 and 40), whose descriptions now say which reward they go with.

In [ ]:
print(c.event_descriptions(f).query("label.str.contains('condition')").to_string(index=False))
mid, nxt = trials.start.values, trials.next_start.values
for lab in ["condition 1", "condition 2", "left condition", "right condition"]:
    trials[lab] = c._any_in(np.sort(events[lab]), mid, nxt)
print(pd.crosstab([trials["left condition"], trials["right condition"]], [trials["condition 1"], trials["condition 2"]]))
s = trials[trials.success]
print(pd.crosstab(s.cls, [s["left condition"], s["condition 1"]]))
print("event labels naming top, bottom, up or down:",
      [k for k in events if any(w in k.lower() for w in ["top", "bottom", "up", "down"])])

Every trial holds one reward condition code, and in the successful trials condition 1 always gives the large
reward and condition 2 the small one, so the reward condition is now described. The left and right
conditions still occur with both reward conditions, and no event codes the target's vertical position, so
the top or bottom position that the paper says sets the reward is still not in the dandiset.

## Earlier findings 11 and 12

> The raw FSCV minute files are joined without their gaps (328 min of scans against 302 min of trials, placed from a single starting time), and the trial-aligned oxidation current cannot be located in them, so the raw FSCV's clock could not be established.

> Only one session of monkey T is deposited. The data behind the electrode yields, cultures, drainage volumes and bone growth, and all of monkey J, are not, so eight of the thirteen claims cannot be tested.

These are reported by `choi-2025-02`: the raw FSCV scan times and the trial-aligned oxidation current
against the raw scans in its findings 2 and 9, checked in `reproduce_choi_2025.ipynb`, and the missing
session, electrode, surgical and monkey J data in its finding 1.